In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit
import scipy.constants as pc
import astropy.units as u
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.table import vstack
from astropy.cosmology import FlatLambdaCDM
from scipy.stats import ttest_rel
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import AutoMinorLocator
import os
from matplotlib.gridspec import GridSpec

FIG_DIR = "/raid/scratch/work/alberttg/Masters_project/Plots"

DATA_PRODUCTS_PATH = "/raid/scratch/work/alberttg/Masters_project/Data_products"

POSTERIOR_PATH = '/raid/scratch/work/austind/GALFIND_WORK/Bagpipes/pipes/posterior'

In [ ]:
def get_posterior(sample_table, SFH_model):
    """
    Finding the paths to the stellar mass posterior, based on galaxy survey id, survey and SFH model used in BAGPIPES.
    """

    surveys = np.asarray(sample_table["SURVEY"].data, dtype=str)

    survey_ids = np.asarray(sample_table["#ID"].data, dtype=int)

    for i in range(len(sample_table)):

        id = survey_ids[i]
        survey = surveys[i]

        if survey in ["CEERSP1", "CEERSP2", "CEERSP3", "CEERSP4", "CEERSP5", "CEERSP6", "CEERSP7", "CEERSP8", "CEERSP9", "CEERSP10"]:

            path = os.path.join(POSTERIOR_PATH, f"v14_psfmatch_F444W_empirical/{survey}/ACS_WFC+NIRCam/0.32as/{SFH_model}/{id}.h5")

        elif survey in ["PRIMER-COSMOS","PRIMER-UDS"]:
                
            path = os.path.join(POSTERIOR_PATH, f"v12_psfmatch_F444W_empirical/{survey}/ACS_WFC+NIRCam/0.32as/{SFH_model}/{id}.h5")


        elif survey in ["JADES-DR3-GS-East", "JADES-DR3-GS-North", "JADES-DR3-GS-South", "JADES-DR3-GS-West", "JADES-DR3-GN-Parallel", "JADES-DR3-GN-Medium", "JADES-DR3-GN-Deep"]:

            path = os.path.join(POSTERIOR_PATH, f"v13_psfmatch_F444W_empirical/{survey}/ACS_WFC+NIRCam/0.32as/{SFH_model}/{id}.h5")
            
        else:
            print(f"Galaxy {id} could not be found in these surveys")
            print(f"Check survey {survey}")
            break

    

In [41]:
def grid_spec_uncertainties(sample_table, n_mass_bins=5):
    """
    """

    redshift = sample_table["input_redshift"].data
     
    mass = sample_table["stellar_mass_50"].data

    mass_errors = sample_table["stellar_mass_84"].data - mass # This is a place holder please change!!!!!!

    z_bins = [[6.5, 7.5], [7.5, 8.5], [8.5, 9.5], [9.5, 10.5], [10.5, 11.5], [11.5, 13.5]]

    fig, axes = plt.subplots(4, 6, figsize=(20,16), constrained_layout=False)
    # Starting binning, bin by mass first (vertical bins)
    # mass_bins = np.linspace(mass.min(), mass.max(), n_mass_bins + 1)
    # mass_bins = [[mass_bins[i], mass_bins[i + 1]] for i in range(len(mass_bins) - 1)]
    # mass_bins.reverse()
    #print("Mass bins: ", mass_bins)
    mass_bins = [[9,11], [8,9], [7,8], [6,7]]

    for i in range(len(mass_bins)):
    
        bin_width = mass_bins[i][1] - mass_bins[i][0]

        mass_bin_mask = (mass >= mass_bins[i][0]) & (mass < mass_bins[i][1]) # Splitting into mass bins

        redshift_m_binned = redshift[mass_bin_mask]
        mass_errors_m_binned = mass_errors[mass_bin_mask]
    
        # Now bin by z (horizontal bins)
        for j in range(len(z_bins)):

            z_bin_mask = (redshift_m_binned >= z_bins[j][0]) & (redshift_m_binned < z_bins[j][1])

            mass_errors_z_binned = mass_errors_m_binned[z_bin_mask] # x axis values now binned by z and mass

            ax = axes[i,j]
        
            # ax.set_ylabel(r'Probability density \left( $\sigma_{M_\star}^{-1}$ \right)', fontsize=16)
            if j == 0:
                ax.set_ylabel(rf"${mass_bins[j][0]} ≤ \log_10(M_\star/M_\odot) < {mass_bins[j][1]}$", fontsize=16)
            
            ax.grid(True)

            ax.tick_params(
                        axis="both",
                        which="major",
                        labelsize=14,
                        length=8,
                        width=1.5,
                        direction="in",
                        top=True,
                        right=True
                    )
            
            ax.xaxis.set_minor_locator(AutoMinorLocator(5))
            ax.yaxis.set_minor_locator(AutoMinorLocator(5))

            ax.tick_params(
                axis="both",
                which="minor",
                length=4,
                width=1,
                direction="in",
                top=True,
                right=True
            )

            ax.hist(mass_errors_z_binned, bins=20, color='steelblue', edgecolor='black') # Try to normalise????
            #N = len(mass_errors_binned)

            legend_text = rf"${z_bins[j][0]} ≤ z < {z_bins[j][1]}$"
            ax.text(0.1, 0.95, legend_text, transform=ax.transAxes, fontsize=15, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
    
    fig.supylabel(r'Galaxy Stellar Mass $\log_{10}(M_\star/M_\odot)$', fontsize=19)
    fig.supxlabel(r'Stellar Mass Error $\sigma_{M_\star}$', fontsize=19)
    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, "gridspec_mass_errors.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    

In [ ]:
if __name__ == "__main__":

    with fits.open('/raid/scratch/work/alberttg/Masters_project/Data_products/First_sample_table.fits') as hdu:

        sample_table = hdu[1].data
    sample_table = Table(sample_table)

    SFH_models = ["Bagpipes_sfh_cont_bursty_zEAZYsfhzblueagn_3.0,10.0Myr_Calzetti_log_10_Z_log_10_BPASS_zfix"]

    grid_spec_uncertainties(sample_table)

